# Lab 01: Building a ReAct Agent from Scratch

## 🎯 What We're Building

In this lab, you will build a **working AI Agent** in three stages:

| Stage | What | Lines of Code | What You Learn |
|-------|------|---------------|----------------|
| **Stage 1** | Raw agent with just OpenAI SDK | ~80 lines | How agents REALLY work under the hood |
| **Stage 2** | Same agent with LangGraph | ~15 lines | Why frameworks exist and what they give you |
| **Stage 3** | Add memory and persistence | ~25 lines | How agents remember and survive crashes |

> **Prerequisites:** Read the [README.md](README.md) first to understand the concepts.

---

## 🔧 Setup

First, let's install dependencies and set up our environment.

In [ ]:
# Install dependencies (run once)
%pip install openai langchain-openai langgraph python-dotenv rich --quiet

In [ ]:
import os
from dotenv import load_dotenv

# Load API key from .env file (or set it directly)
load_dotenv("../.env")

# Verify the key is loaded
api_key = os.getenv("OPENAI_API_KEY")
if not api_key or api_key == "sk-your-key-here":
    print("⚠️  Please set your OPENAI_API_KEY in the ../.env file!")
    print("   Copy ../.env.template to ../.env and add your key.")
else:
    print(f"✅ API key loaded (ends with ...{api_key[-4:]})")

---

# 🏗️ STAGE 1: Build a Raw Agent (No Framework)

We're going to build a complete agent using **only** the OpenAI Python SDK.
No LangChain, no LangGraph — just raw Python.

**Why start here?** Because you need to see every moving part before a framework hides them.

---

## Step 1.1: Define Our Tools

An agent needs tools — functions it can call to interact with the world.
Let's create three simple tools:

| Tool | Purpose | Example |
|------|---------|--------|
| `get_weather` | Get weather for a city | "Tel Aviv" → "25°C, sunny" |
| `calculate` | Do math | "15 * 7 + 3" → 108 |
| `search_knowledge` | Search a knowledge base | "vacation policy" → relevant info |

These are intentionally simple — we're learning the **agent pattern**, not building real integrations.

In [ ]:
import json

# ============================================================
# TOOL 1: Weather (simulated)
# In production, this would call a real weather API.
# ============================================================
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated weather data
    weather_data = {
        "tel aviv": {"temp": 28, "condition": "sunny", "humidity": 65},
        "new york": {"temp": 12, "condition": "cloudy", "humidity": 70},
        "london":   {"temp": 8,  "condition": "rainy",  "humidity": 85},
        "tokyo":    {"temp": 20, "condition": "clear",  "humidity": 55},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown", "humidity": 50})
    return f"Weather in {city}: {data['temp']}°C, {data['condition']}, humidity {data['humidity']}%"


# ============================================================
# TOOL 2: Calculator
# Uses Python's eval (safe for math expressions only)
# ============================================================
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    # Only allow safe math characters
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return "Error: only mathematical expressions are allowed"
    try:
        result = eval(expression)  # Safe because we validated input
        return f"Result: {expression} = {result}"
    except Exception as e:
        return f"Error calculating: {e}"


# ============================================================
# TOOL 3: Knowledge Base Search
# A tiny in-memory knowledge base (in production: vector DB)
# ============================================================
KNOWLEDGE_BASE = [
    {"topic": "vacation policy", "content": "Employees get 22 vacation days per year. Unused days can carry over up to 5 days. Must request vacation at least 2 weeks in advance."},
    {"topic": "remote work", "content": "Employees can work remotely up to 3 days per week. Must be available on Teams during core hours (10am-4pm). Manager approval needed for full remote."},
    {"topic": "expense policy", "content": "Meals up to $50/day during travel. Flights must be economy class. Hotel max $200/night. All expenses need receipts and manager approval within 30 days."},
    {"topic": "onboarding", "content": "New employees have a 90-day onboarding period. Week 1: HR orientation. Week 2-4: Team training. Month 2-3: Shadowing and first projects."},
]

def search_knowledge(query: str) -> str:
    """Search the company knowledge base for relevant information."""
    query_lower = query.lower()
    results = []
    for item in KNOWLEDGE_BASE:
        # Simple keyword matching (in production: vector similarity search)
        if any(word in query_lower for word in item["topic"].split()):
            results.append(f"[{item['topic']}]: {item['content']}")
    
    if results:
        return "\n\n".join(results)
    return "No relevant information found in the knowledge base."


# Test our tools
print("🌤️", get_weather("Tel Aviv"))
print("🔢", calculate("15 * 7 + 3"))
print("📚", search_knowledge("vacation days"))

## Step 1.2: Define Tool Schemas for the LLM

The LLM doesn't call our Python functions directly.
Instead, we describe each tool as a **JSON schema**, and the LLM outputs
JSON telling us which tool to call and with what arguments.

Think of it like a restaurant menu — we show the LLM the menu, and it tells us what to order.

In [ ]:
# Tool schemas — this is what we send to the LLM so it knows what tools exist
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city. Use this when the user asks about weather, temperature, or conditions in a specific location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'Tel Aviv', 'New York'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use this for any math calculations the user needs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression, e.g., '15 * 7 + 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search the company knowledge base for information about policies, procedures, and guidelines. Use this when the user asks about company rules, policies, or internal information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query describing what information to look for"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Map tool names to actual Python functions
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_knowledge": search_knowledge,
}

print(f"✅ Defined {len(TOOL_SCHEMAS)} tools: {', '.join(TOOL_FUNCTIONS.keys())}")

## Step 1.3: Build the ReAct Loop

This is the **heart of every agent**. Here's what the loop does:

```
1. Send messages + tool schemas to LLM
2. LLM responds with either:
   a) A final text answer → we're done!
   b) A tool_call request → execute the tool, add result to messages, go to step 1
3. Repeat until the LLM gives a text answer OR we hit max iterations
```

**Pay close attention to the message format** — understanding how messages flow is key to understanding agents.

In [ ]:
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

console = Console()
client = OpenAI()  # Uses OPENAI_API_KEY from environment

MODEL = "gpt-4o-mini"  # Cheap and fast — perfect for learning

SYSTEM_PROMPT = """You are a helpful assistant with access to tools.
Use the available tools when you need to look up information, do calculations,
or check weather. Always explain your reasoning."""


def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    Run a ReAct agent loop.
    
    This is the ENTIRE agent — ~40 lines of core logic.
    Everything else is just printing for visibility.
    """
    # Initialize the message history
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    console.print(f"\n[bold blue]👤 User:[/] {user_message}\n")
    
    for iteration in range(1, max_iterations + 1):
        console.print(f"[dim]--- Iteration {iteration}/{max_iterations} ---[/]")
        
        # ===== STEP 1: Call the LLM =====
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        
        assistant_message = response.choices[0].message
        
        # ===== STEP 2: Check if the LLM wants to call a tool =====
        if assistant_message.tool_calls:
            # The LLM wants to use a tool!
            # Add the assistant's message (with tool_calls) to history
            messages.append(assistant_message)
            
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                
                console.print(f"  [yellow]🤔 Think:[/] I need to use [bold]{tool_name}[/]")
                console.print(f"  [green]🔧 Act:[/]   {tool_name}({tool_args})")
                
                # ===== STEP 3: Execute the tool =====
                tool_function = TOOL_FUNCTIONS.get(tool_name)
                if tool_function:
                    # Call the ACTUAL Python function
                    result = tool_function(**tool_args)
                else:
                    result = f"Error: Unknown tool '{tool_name}'"
                
                console.print(f"  [cyan]👀 Observe:[/] {result}\n")
                
                # ===== STEP 4: Add the tool result to message history =====
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        else:
            # No tool calls — the LLM is giving us the final answer!
            final_answer = assistant_message.content
            console.print(f"[bold green]📤 Answer:[/] {final_answer}\n")
            
            # Show stats
            console.print(f"[dim]Completed in {iteration} iteration(s), "
                         f"{len(messages)} messages in history[/]")
            return final_answer
    
    return "⚠️ Max iterations reached without a final answer."


print("✅ Agent function defined. Let's test it!")

## Step 1.4: Test the Agent!

Now let's run our agent with different types of questions.

**Watch carefully:** For each question, observe:
- How many iterations the agent takes
- Which tools it chooses
- How it combines information from multiple tool calls

In [ ]:
# Test 1: Simple question that needs ONE tool
run_agent("What's the weather like in Tel Aviv?")

In [ ]:
# Test 2: Question that needs a DIFFERENT tool
run_agent("What is our company's vacation policy?")

In [ ]:
# Test 3: Question that needs MULTIPLE tools!
# Watch how the agent decides to call multiple tools and combines results
run_agent("I'm planning a trip to Tokyo. What's the weather there? "
          "Also, what's the daily meal expense limit? "
          "If I eat 3 meals at $15 each, am I within budget?")

In [ ]:
# Test 4: Question that needs NO tools
# The agent should just answer directly — no tool calls
run_agent("What is 2 + 2? Don't use any tools, just answer.")

### 🤔 Reflection: What Did We Learn?

Take a moment to answer these questions:

1. **Who decides which tool to use?** → The LLM, based on the tool descriptions and the user's question
2. **Who executes the tool?** → Our Python code, NOT the LLM
3. **How does the LLM know the tool result?** → We send it back as a `tool` message
4. **When does the loop stop?** → When the LLM responds with text instead of a tool_call
5. **What prevents infinite loops?** → Our `max_iterations` safety check

This is **the core of every agent in every framework**. LangChain, LangGraph, Semantic Kernel — they all do exactly this under the hood.

---

## Step 1.5: Inspect the Message History

Let's look at the actual messages that flow between our code and the LLM.
Understanding this is **critical** — it's how agents maintain context.

In [ ]:
from IPython.display import HTML, display

def run_agent_with_history(user_message: str, max_iterations: int = 5):
    """Same agent, but returns the full message history for inspection."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]
    
    for iteration in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS
        )
        msg = response.choices[0].message
        
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                fn = TOOL_FUNCTIONS.get(tc.function.name)
                result = fn(**args) if fn else "Unknown tool"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            messages.append({"role": "assistant", "content": msg.content})
            break
    
    return messages

# Run and inspect
history = run_agent_with_history("What's the weather in London and what is 32 * 9?")

# Display the message flow
print(f"\n📬 Total messages in history: {len(history)}\n")
for i, msg in enumerate(history):
    role = msg["role"] if isinstance(msg, dict) else msg.role
    
    if role == "system":
        print(f"  [{i}] 🔧 SYSTEM: (system prompt)")
    elif role == "user":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"  [{i}] 👤 USER: {content[:80]}")
    elif role == "assistant":
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [{i}] 🤖 ASSISTANT: tool_call → {tc.function.name}({tc.function.arguments})")
        else:
            content = msg["content"] if isinstance(msg, dict) else msg.content
            print(f"  [{i}] 🤖 ASSISTANT: {content[:80]}...")
    elif role == "tool":
        content = msg["content"] if isinstance(msg, dict) else msg.content
        print(f"  [{i}] 🔧 TOOL RESULT: {content[:80]}")

### 💡 Key Insight: The Message History IS the Agent's Memory

Look at the message flow above. Every message stays in the history:

```
[0] system    →  "You are a helpful assistant..."
[1] user      →  "What's the weather in London and what is 32 * 9?"
[2] assistant →  tool_call: get_weather({city: "London"})
[3] tool      →  "Weather in London: 8°C, rainy, humidity 85%"
[4] assistant →  tool_call: calculate({expression: "32 * 9"})
[5] tool      →  "Result: 32 * 9 = 288"
[6] assistant →  "The weather in London is 8°C and rainy. 32 × 9 = 288."
```

The LLM sees **all previous messages** on every call. That's how it knows:
- What the user asked
- What tools it already called
- What results it got
- What's left to do

**This is the agent's "short-term memory"** — it only lasts for one request.

---

# 🏗️ STAGE 2: Rebuild with LangGraph

Now let's build the **exact same agent** using LangGraph.

You'll immediately see why frameworks exist — the same agent takes **~15 lines** instead of ~80.

### What is LangGraph?

LangGraph models agents as **graphs**:
- **Nodes** = functions (call LLM, execute tools)
- **Edges** = transitions (what to do next)
- **State** = shared data that flows through the graph

```
┌─────────────────────────────────────────────────────────────┐
│                  LangGraph Agent                            │
│                                                             │
│   ▶ START ──▶ [Agent Node] ──▶ Should I use a tool?        │
│                    ▲               │          │             │
│                    │          Yes  │     No   │             │
│                    │               ▼          ▼             │
│                    └── [Tool Node]      ⏹ END              │
│                                                             │
│   The framework handles the loop automatically!            │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ============================================================
# Step 2.1: Define tools using LangChain's @tool decorator
# This is MUCH simpler than writing JSON schemas by hand!
# ============================================================

@tool
def get_weather_lc(city: str) -> str:
    """Get the current weather for a given city.
    Use this when the user asks about weather or temperature."""
    return get_weather(city)  # Reuse our function from Stage 1

@tool
def calculate_lc(expression: str) -> str:
    """Evaluate a mathematical expression.
    Use this for any math calculations."""
    return calculate(expression)  # Reuse our function from Stage 1

@tool
def search_knowledge_lc(query: str) -> str:
    """Search the company knowledge base for policies and information.
    Use this when the user asks about company rules or guidelines."""
    return search_knowledge(query)  # Reuse our function from Stage 1

# ============================================================
# Step 2.2: Create the agent — THIS IS THE ENTIRE AGENT!
# Compare this to the ~80 lines we wrote in Stage 1.
# ============================================================

llm = ChatOpenAI(model="gpt-4o-mini")
tools = [get_weather_lc, calculate_lc, search_knowledge_lc]

# One line to create a complete ReAct agent with tool loop:
agent = create_react_agent(llm, tools)

print("✅ LangGraph agent created with 3 tools")
print(f"   Tools: {[t.name for t in tools]}")

In [ ]:
# Step 2.3: Run the LangGraph agent
# Same questions as Stage 1 — same results, way less code

result = agent.invoke(
    {"messages": [("user", "What's the weather in Tel Aviv?")]}
)

# Print all messages to see the flow
print("📬 LangGraph message flow:\n")
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  🤖 {role}: tool_call → {tc['name']}({tc['args']})")
    elif hasattr(msg, 'content') and msg.content:
        prefix = "👤" if "Human" in role else "🤖" if "AI" in role else "🔧"
        print(f"  {prefix} {role}: {msg.content[:100]}")

In [ ]:
# Multi-tool question — same as Stage 1 Test 3
result = agent.invoke({
    "messages": [(
        "user", 
        "I'm planning a trip to Tokyo. What's the weather there? "
        "Also, what's the daily meal expense limit? "
        "If I eat 3 meals at $15 each, am I within budget?"
    )]
})

# Just print the final answer
final = result["messages"][-1].content
print(f"📤 Final answer:\n\n{final}")

### 🤔 Stage 1 vs Stage 2: What Changed?

| Aspect | Stage 1 (Raw Python) | Stage 2 (LangGraph) |
|--------|---------------------|----------------------|
| **Lines of code** | ~80 | ~15 |
| **ReAct loop** | We wrote it manually | Framework handles it |
| **Tool schemas** | JSON by hand | `@tool` decorator auto-generates |
| **Message management** | We managed the list | Framework manages state |
| **Error handling** | We had to add it | Built-in |
| **Max iterations** | We implemented it | Built-in |

**The framework does exactly what we built by hand** — but it's tested, handles edge cases, and is production-ready.

---

# 🏗️ STAGE 3: Add Memory and Persistence

Our agent has a problem: **it forgets everything between requests.**

Try this:

In [ ]:
# The agent has NO memory — it doesn't know about previous conversations
result1 = agent.invoke({"messages": [("user", "My name is Roi.")]})
print("Response 1:", result1["messages"][-1].content)

result2 = agent.invoke({"messages": [("user", "What's my name?")]})
print("Response 2:", result2["messages"][-1].content)
print("\n😔 It doesn't remember! Each call starts fresh.")

### Adding Thread-Based Memory

LangGraph solves this with **checkpointing** — after each interaction,
the agent's state is saved. Next time the same `thread_id` is used,
the previous conversation is loaded.

Think of it like a chat app — each thread remembers the conversation.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create agent WITH memory (checkpointing)
memory = MemorySaver()  # In-memory (for production, use SQLite/Postgres)
agent_with_memory = create_react_agent(llm, tools, checkpointer=memory)

# Use a thread_id to identify the conversation
config = {"configurable": {"thread_id": "user-roi-123"}}

# First message
result1 = agent_with_memory.invoke(
    {"messages": [("user", "My name is Roi. I work in the analytics team.")]},
    config=config
)
print("Response 1:", result1["messages"][-1].content)

# Second message — SAME thread_id
result2 = agent_with_memory.invoke(
    {"messages": [("user", "What's my name and which team do I work in?")]},
    config=config
)
print("\nResponse 2:", result2["messages"][-1].content)

print("\n🎉 It remembers! The thread_id connects the conversations.")

In [ ]:
# Different thread_id = different conversation = no memory
config_new = {"configurable": {"thread_id": "different-user-456"}}

result3 = agent_with_memory.invoke(
    {"messages": [("user", "What's my name?")]},
    config=config_new
)
print("Different thread:", result3["messages"][-1].content)
print("\n🔒 Different thread = isolated conversation. This is multi-tenancy!")

---

# 🎓 Summary: What We Built and Learned

### The Three Stages

```
Stage 1                   Stage 2                   Stage 3
Raw Python                LangGraph                 + Memory
━━━━━━━━━━                ━━━━━━━━━                ━━━━━━━━
                                                    
✅ Built ReAct loop       ✅ Same agent,            ✅ Agent remembers
   from scratch              15 lines                  conversations
✅ Managed messages       ✅ Framework handles      ✅ Thread isolation
   manually                  everything                (multi-tenant)
✅ Parsed tool calls      ✅ @tool decorator        ✅ Checkpointing
   from JSON                 auto-generates            saves state
                                                    
~80 lines of code         ~15 lines of code         ~25 lines of code
```

### Key Concepts Learned

| Concept | What It Means |
|---------|---------------|
| **Agent** | LLM + Tools + Loop |
| **ReAct** | Think → Act → Observe → repeat |
| **Tool Calling** | LLM outputs JSON, YOUR code executes the tool |
| **Message History** | The agent's short-term memory within a request |
| **Max Iterations** | Safety limit to prevent infinite loops |
| **LangGraph** | Framework that models agents as graphs (nodes + edges) |
| **Checkpointing** | Saving agent state for memory across requests |
| **Thread ID** | Identifier that separates conversations (multi-tenancy) |

### What's Next?

In **Lab 02**, we'll add **smart model routing** — using cheap models for simple tasks and expensive models for complex ones. You'll see how this can cut costs by 80%.

---

> **[← Back to Labs Overview](../README.md)** | **[→ Lab 02: Smart Model Routing](../lab-02-model-routing/README.md)**